In [1]:
# Cell 1: Imports and Setup

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from datetime import datetime

# Add project root to path
project_root = Path('.').absolute().parent.parent.parent
sys.path.insert(0, str(project_root))

from new_pipeline.models.ros import (
            prepare_ros_training_data,
            HitterROSEnsemble,
            ROS_HITTER_FEATURES,
    temporal_cv_split,
    calculate_ros_metrics
)
from new_pipeline.common.features import ROSFeatureBuilder
from new_pipeline.common.data_preparation import create_multipoint_splits
from new_pipeline.notebooks.shared.pipeline_runner import load_historical_data, load_current_season_data, run_data_pipeline

print("Imports successful!")
print(f"ROS Hitter Features: {len(ROS_HITTER_FEATURES)}")

20:23:04 - new_pipeline.common.logging_config - INFO - Logging module initialized for new_pipeline
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\fs\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful!
ROS Hitter Features: 66


In [2]:
# Cell 2: Load and Process Historical Data (2016-2024)

print("="*70)
print("LOADING HISTORICAL FULL-SEASON DATA (2016-2024)")
print("="*70)

from new_pipeline.notebooks.shared.pipeline_runner import run_data_pipeline

# Load raw historical data
print("\nLoading raw hitter data...")
hitter_raw = load_historical_data(
    player_type='hitter',
    years=range(2016, 2025)  # 2016-2024
)

print(f"\nLoaded {len(hitter_raw)} hitter seasons (raw)")

# Process through pipeline (adds features + Age + WAR_per_600)
print("\nProcessing hitter data through pipeline...")
print("  Pipeline adds: Age (from BP_Data), features, WAR_per_600")
hitter_processed = run_data_pipeline(hitter_raw, 'hitter')

print(f"\nProcessed {len(hitter_processed)} qualified hitters")
print(f"Years: {sorted(hitter_processed['Year'].unique())}")

# Verify Age and WAR rates are present
print("\nData quality checks:")
print(f"  Hitters with Age: {hitter_processed['Age'].notna().sum()}")
print(f"  Hitters with WAR_per_600: {hitter_processed['WAR_per_600'].notna().sum()}")

if 'Age' in hitter_processed.columns:
    print(f"  Hitter Age range: [{hitter_processed['Age'].min():.0f}, {hitter_processed['Age'].max():.0f}]")

20:23:11 - new_pipeline.common.transformers.filters - INFO - PAFilter: Removed 1524 hitters with < 75 PA (full season)


LOADING HISTORICAL FULL-SEASON DATA (2016-2024)

Loading raw hitter data...

Loaded 5760 hitter seasons (raw)

Processing hitter data through pipeline...
  Pipeline adds: Age (from BP_Data), features, WAR_per_600


20:23:12 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 2088 hitters
20:23:12 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 20-45)
20:23:12 - new_pipeline.common.transformers.hitter_features - INFO - Loading hitter features...
20:23:17 - new_pipeline.common.transformers.hitter_features - INFO - Loaded 11 hitter feature sets (33 total columns)
20:23:17 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 28 features
20:23:17 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 70 missing values
20:23:17 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'K%' range [3.09, 51.25] outside expected [0, 50]
  - Feature 'AVG' range [0.09, 0.38] outside expected [0.1, 0.4]
  - Feature 'OBP' range [0.10, 0.49] outside expected [0.2, 0.5]
  - Feature 'SLG' range [0.13, 0.73] ou


Processed 4236 qualified hitters
Years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Data quality checks:
  Hitters with Age: 4236
  Hitters with WAR_per_600: 4236
  Hitter Age range: [20, 45]


In [3]:
# Cell 2.5: Load Injury Data (Auto-discover historical years)

from new_pipeline.common.data_preparation.injury_data_loader import load_injury_data_multiple_years

print("="*70)
print("LOADING INJURY DATA")
print("="*70)

try:
    # Construct absolute path using project_root from Cell 1
    injury_data_dir = project_root / "MLB Player Data" / "FanGraphs_Data" / "injuries"
    
    # Auto-discover all historical years (excludes current season for training)
    injury_data_historical = load_injury_data_multiple_years(
        data_dir=str(injury_data_dir),
        auto_discover=True,
        exclude_current_season=True
    )
    
    print(f"Injury data summary:")
    print(f"  Total records: {len(injury_data_historical)}")
    print(f"  Unique players: {injury_data_historical['MLBAMID'].nunique()}")
    print(f"  Years covered: {sorted(injury_data_historical['Year'].unique())}")
    
    if "status" in injury_data_historical.columns:
        print(f"  Status breakdown: {injury_data_historical['status'].value_counts().to_dict()}")
    
    print("Injury data loaded successfully!")
    
except Exception as e:
    print(f"ERROR: Failed to load injury data")
    print(f"  {type(e).__name__}: {e}")
    raise


LOADING INJURY DATA
Discovered injury data years: [2020, 2021, 2022, 2023, 2024]
Loaded 530 injury records for 2020
  Status breakdown: {'Active roster': 253, '45-Day IL': 114, '10-Day IL': 86, 'Player Pool': 20, 'Released': 3, '7-Day IL': 3, 'Outrighted': 2, 'Opted out of season': 1, 'Designated for assignment': 1}
Loaded 1221 injury records for 2021
  Status breakdown: {'Activated': 958, '60-Day IL': 168, '10-Day IL': 81, 'CV-19 IL': 13}
Loaded 981 injury records for 2022
  Status breakdown: {'Activated': 727, '60-Day IL': 160, '15-Day IL': 56, '10-Day IL': 33, 'CV-19 IL': 3, '7-Day IL': 2}
Loaded 856 injury records for 2023
  Status breakdown: {'Activated': 597, '60-Day IL': 170, '15-Day IL': 58, '10-Day IL': 30, '7-Day IL': 1}
Loaded 775 injury records for 2024
  Status breakdown: {'Activated': 775}

Combined injury data: 4363 total records across 5 years
Injury data summary:
  Total records: 4363
  Unique players: 1657
  Years covered: [np.int64(2020), np.int64(2021), np.int64(202

In [4]:
# Cell 3: Create Multipoint Season Splits

print("="*70)
print("CREATING MULTIPOINT SEASON SPLITS")
print("="*70)

# Create splits at 25%, 50%, 75% of season
# Use PROCESSED data (has Age, WAR_per_600, all features)
split_points = [0.25, 0.5, 0.75]

print(f"\nCreating splits at: {split_points}")
print("Each player-season becomes 3 training samples")
print("  - 25% split: 'At 40 games, predict remaining 122 games WAR'")
print("  - 50% split: 'At 81 games, predict remaining 81 games WAR'")
print("  - 75% split: 'At 121 games, predict remaining 41 games WAR'")

print("\nSplitting hitter data (from PROCESSED - has Age + WAR_per_600)...")
hitter_splits = create_multipoint_splits(
    full_season_df=hitter_processed,  # Use processed data
    split_points=split_points,
    player_type='hitter',
    season_length=162
)

print(f"\nCreated {len(hitter_splits)} hitter split samples")
print(f"  Original seasons: {len(hitter_processed)}")
print(f"  Splits per season: {len(split_points)}")
print(f"\nSample split row (hitter):")
print(hitter_splits.head(1).T)

CREATING MULTIPOINT SEASON SPLITS

Creating splits at: [0.25, 0.5, 0.75]
Each player-season becomes 3 training samples
  - 25% split: 'At 40 games, predict remaining 122 games WAR'
  - 50% split: 'At 81 games, predict remaining 81 games WAR'
  - 75% split: 'At 121 games, predict remaining 41 games WAR'

Splitting hitter data (from PROCESSED - has Age + WAR_per_600)...

Created 12702 hitter split samples
  Original seasons: 4236
  Splits per season: 3

Sample split row (hitter):
                                84
playerid                    120074
Name                   David Ortiz
Team                           BOS
Year                          2016
split_point                   0.25
season_completion_pct         0.25
team_games_played             40.5
current_G                    37.75
full_G                         151
current_PA                   156.5
full_PA                        626
current_WAR               1.141349
full_WAR                  4.565398
MLBAMID                    

In [5]:
# Cell 4: Build ROS Features

print("="*70)
print("BUILDING ROS FEATURES")
print("="*70)

hitter_builder = ROSFeatureBuilder(player_type='hitter')

print("\nBuilding hitter ROS features from splits...")
hitter_with_features = hitter_builder.build_features_batch(
    current_season_df=hitter_splits,  # Multipoint split data (current stats)
    historical_df=hitter_processed,   # Full season data (for baselines, age curves, etc.)
    injury_df=injury_data_historical,  # Historical injury data
    current_date=None  # Use latest injury data available
)

print(f"\nBuilt features for {len(hitter_with_features)} hitter samples")
print(f"Feature columns: {len(hitter_with_features.columns)}")

# Verify key feature columns are present
from new_pipeline.models.ros import ROS_HITTER_FEATURES
print(f"\nROS Hitter Features: {len(ROS_HITTER_FEATURES)}")

hitter_feature_count = len([col for col in hitter_with_features.columns if col in ROS_HITTER_FEATURES])
print(f"Found {hitter_feature_count}/{len(ROS_HITTER_FEATURES)} ROS features")

# Check for missing features
missing_features = [col for col in ROS_HITTER_FEATURES if col not in hitter_with_features.columns]
if missing_features:
    print(f"\nWARNING: Missing features: {missing_features}")

BUILDING ROS FEATURES

Building hitter ROS features from splits...

Built features for 12702 hitter samples
Feature columns: 147

ROS Hitter Features: 66
Found 66/66 ROS features


In [6]:
# Cell 5: Prepare Training Data

print("="*70)
print("PREPARING TRAINING DATA")
print("="*70)

print("\nPreparing hitter training data...")
hitter_train_df, X_hitter, y_hitter = prepare_ros_training_data(
    multipoint_df=hitter_with_features,  # From Cell 4 (has all features)
    feature_columns=ROS_HITTER_FEATURES,
    target_column='remaining_WAR'
)

print(f"\n{'='*70}")
print("TRAINING DATA SUMMARY")
print(f"{'='*70}")

print(f"\nHitters:")
print(f"  Samples: {len(X_hitter)}")
print(f"  Features: {X_hitter.shape[1]} (expected: {len(ROS_HITTER_FEATURES)})")
print(f"  Target (remaining WAR):")
print(f"    Mean: {y_hitter.mean():.2f}")
print(f"    Std: {y_hitter.std():.2f}")
print(f"    Range: [{y_hitter.min():.2f}, {y_hitter.max():.2f}]")

# Verify feature counts match
assert X_hitter.shape[1] == len(ROS_HITTER_FEATURES), f"Hitter feature mismatch: {X_hitter.shape[1]} != {len(ROS_HITTER_FEATURES)}"

print(f"\n{'='*70}")
print("READY FOR TRAINING")
print(f"{'='*70}")

PREPARING TRAINING DATA

Preparing hitter training data...

TRAINING DATA SUMMARY

Hitters:
  Samples: 12702
  Features: 66 (expected: 66)
  Target (remaining WAR):
    Mean: 0.58
    Std: 0.96
    Range: [-1.91, 8.50]

READY FOR TRAINING


In [7]:
# Cell 6: Train Hitter ROS Ensemble

print("="*70)
print("TRAINING HITTER ROS ENSEMBLE")
print("="*70)

# Initialize ensemble with validated weights
print("\nInitializing HitterROSEnsemble...")
hitter_ros = HitterROSEnsemble(
    weights=[0.5, 0.4, 0.1],  # DirectROSForecaster, DartsTemporalEnsemble, Baseline
    feature_columns=ROS_HITTER_FEATURES,
    target_column='remaining_WAR'
)

# Train on multipoint historical data
print("\nFitting ensemble on historical data (2016-2024)...")
print("This may take several minutes...")

hitter_ros.fit(
    historical_df=hitter_train_df,
    feature_columns=ROS_HITTER_FEATURES,
    target_column='remaining_WAR'
)

print("\n" + "="*70)
print("HITTER ROS ENSEMBLE TRAINING COMPLETE")
print("="*70)

TRAINING HITTER ROS ENSEMBLE

Initializing HitterROSEnsemble...

Fitting ensemble on historical data (2016-2024)...
This may take several minutes...
Fitting HitterROSEnsemble on 12702 samples...
  Converting to sktime format (DirectROSForecaster)...
  Fitting DirectROSForecaster (11835 samples after validation)...
  Converting to Darts format (Temporal ensemble)...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


  Fitting DartsTemporalEnsemble (509 player series)...


c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_Bias', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_Elite_Bias', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_Elite_MAE', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_MAE', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=A

    Note: AutoARIMA skipped for 509/509 players (<10 years data)
  Preparing baseline training data...
  Fitting baseline MultiQuantileHistGB (12702 samples)...
Ensemble fitting complete.

HITTER ROS ENSEMBLE TRAINING COMPLETE


In [8]:
# Cell 8: Validate Training

print("="*70)
print("VALIDATING TRAINED MODELS")
print("="*70)

# Test hitter predictions on small sample
print("\nTesting hitter ROS predictions...")
sample_size = 10
# Use DataFrame slicing and pass historical data for cascading fallback
hitter_sample_pred = hitter_ros.predict(
    hitter_train_df.iloc[:sample_size],  # DataFrame with playerid column
    hitter_train_df  # Historical data for tier-based predictions
)
print(f"  Sample predictions: {hitter_sample_pred}")
print(f"  Sample actuals:     {y_hitter[:sample_size]}")
print(f"  Prediction shape: {hitter_sample_pred.shape}")


# Verify predictions are reasonable
assert len(hitter_sample_pred) == sample_size, "Hitter prediction failed"
assert not np.isnan(hitter_sample_pred).any(), "Hitter predictions contain NaN"

print("\n" + "="*70)
print("VALIDATION PASSED")
print("="*70)

VALIDATING TRAINED MODELS

Testing hitter ROS predictions...
  Player tiers: Tier1=0, Tier2=4, Tier3=6
  Sample predictions: [ 3.42498334  2.30066492  1.14071462 -0.7263836  -0.54594623 -0.29646899
  3.80462551  2.51658048  1.28227237  2.01548997]
  Sample actuals:     [ 3.42404837  2.28269891  1.14134946 -0.91947216 -0.61298144 -0.30649072
  3.86893033  2.57928689  1.28964344  1.9195067 ]
  Prediction shape: (10,)

VALIDATION PASSED


In [9]:
# Cell 9: Save Trained Models and Historical Split Data

print("="*70)
print("SAVING TRAINED MODELS AND SPLIT DATA")
print("="*70)

# Create models directory
models_dir = project_root / 'models'
models_dir.mkdir(exist_ok=True)

# Save hitter ROS model
hitter_path = models_dir / 'hitter_ros_2025.pkl'
joblib.dump(hitter_ros, hitter_path)
print(f"\nSaved: {hitter_path}")

# Verify file size (trained models should be > 100KB)
hitter_size_mb = hitter_path.stat().st_size / 1024 / 1024
print(f"  File size: {hitter_size_mb:.1f} MB")
assert hitter_size_mb > 0.1, f"Hitter model too small ({hitter_size_mb:.1f} MB) - likely not trained"

# Save hitter Darts temporal models separately (they don't serialize with joblib)
if hitter_ros.temporal_model_fitted:
    print("\nSaving hitter Darts temporal models...")
    hitter_tcn_path = models_dir / 'hitter_ros_tcn_2025.pt'
    hitter_tsmixer_path = models_dir / 'hitter_ros_tsmixer_2025.pt'
    
    hitter_ros.temporal_model.tcn.save(str(hitter_tcn_path))
    hitter_ros.temporal_model.tsmixer.save(str(hitter_tsmixer_path))
    
    print(f"  Saved TCN: {hitter_tcn_path.name}")
    print(f"  Saved TSMixer: {hitter_tsmixer_path.name}")
else:
    print("\n  Note: Hitter temporal model not fitted, skipping Darts model save")


# Verify file size

    
hitter_splits_path = models_dir / 'hitter_splits_2016_2024.pkl'
joblib.dump(hitter_with_features, hitter_splits_path)
print(f"\nSaved: {hitter_splits_path}")

hitter_splits_size_mb = hitter_splits_path.stat().st_size / 1024 / 1024
print(f"  File size: {hitter_splits_size_mb:.1f} MB")
print(f"  Rows: {len(hitter_with_features)}")
print(f"  Columns: {len(hitter_with_features.columns)}")
print(f"  Split points: {sorted(hitter_with_features['split_point'].unique())}")



print("\n" + "="*70)
print("MODELS AND SPLIT DATA SAVED SUCCESSFULLY")
print("="*70)
print("\nSaved files:")
print(f"  - {hitter_path.name} (trained model)")
if hitter_ros.temporal_model_fitted:
    print(f"  - hitter_ros_tcn_2025.pt (Darts TCN model)")
    print(f"  - hitter_ros_tsmixer_2025.pt (Darts TSMixer model)")
print(f"  - {hitter_splits_path.name} (historical splits with remaining_WAR)")
print("\nNext: Load these in oWAR_overview.ipynb for inference")

SAVING TRAINED MODELS AND SPLIT DATA

Saved: c:\Users\nairs\Documents\GithubProjects\oWAR\models\hitter_ros_2025.pkl
  File size: 70.2 MB

Saving hitter Darts temporal models...
  Saved TCN: hitter_ros_tcn_2025.pt
  Saved TSMixer: hitter_ros_tsmixer_2025.pt

Saved: c:\Users\nairs\Documents\GithubProjects\oWAR\models\hitter_splits_2016_2024.pkl
  File size: 14.2 MB
  Rows: 12702
  Columns: 147
  Split points: [np.float64(0.25), np.float64(0.5), np.float64(0.75)]

MODELS AND SPLIT DATA SAVED SUCCESSFULLY

Saved files:
  - hitter_ros_2025.pkl (trained model)
  - hitter_ros_tcn_2025.pt (Darts TCN model)
  - hitter_ros_tsmixer_2025.pt (Darts TSMixer model)
  - hitter_splits_2016_2024.pkl (historical splits with remaining_WAR)

Next: Load these in oWAR_overview.ipynb for inference


In [10]:
# Cell 9.6: Hitter ROS Feature Building & Prediction

# Suppress PyTorch Lightning verbosity for cleaner output
import os
import warnings
import logging

os.environ['PYTORCH_LIGHTNING_VERBOSITY'] = '0'
warnings.filterwarnings('ignore', category=UserWarning, module='pytorch_lightning')
warnings.filterwarnings('ignore', category=FutureWarning, module='pytorch_lightning')
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning.utilities.rank_zero").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning.accelerators.cuda").setLevel(logging.ERROR)

print("="*90)
print("HITTER ROS FEATURE BUILDING & PREDICTION")
print("="*90)
print()

# Reload modules to ensure we have the latest version
import importlib
import new_pipeline.common.projections.usage_projections
import new_pipeline.common.projections.ros_projections
importlib.reload(new_pipeline.common.projections.usage_projections)
importlib.reload(new_pipeline.common.projections.ros_projections)

from new_pipeline.common.projections.usage_projections import (
    get_team_games_from_data,
    calculate_hitter_remaining_pa
)
from new_pipeline.common.projections.ros_projections import format_hitter_ros_display
from new_pipeline.common.data_preparation.clean_multi_team_data import clean_multi_team_players

hitter_2025_raw = load_current_season_data('hitter', year=2025)

# Clean multi-team players
hitter_2025_raw = clean_multi_team_players(hitter_2025_raw, year=2025, player_type="hitter")
print(f"Cleaned hitters - Multi-team players: {sum(hitter_2025_raw['Team'].str.contains(',', na=False))}")
hitter_2025_processed = run_data_pipeline(hitter_2025_raw, player_type='hitter')

# Get team-specific games played using utility function
team_games_dict, league_median_games = get_team_games_from_data(hitter_2025_processed)

# Calculate season progression
season_pct = league_median_games / 162
games_remaining = 162 - league_median_games

print(f"League median games: {league_median_games:.0f}, Remaining: {games_remaining:.0f}, Season: {season_pct:.1%}")
print(f"(Individual projections use team-specific games + multi-team handling)")
print()

# Get ALL qualified hitters (minimum PA threshold)
# Use 3.1 PA/G * 95 games = ~75 PA minimum
min_pa = 75
all_qualified_hitters = hitter_2025_processed[hitter_2025_processed['PA'] >= min_pa].copy()

print(f"Building ROS features for ALL qualified hitters (>={min_pa} PA)...")
print(f"  Found {len(all_qualified_hitters)} qualified hitters")

# Build ROS features for ALL qualified hitters
hitter_ros_features_all = hitter_builder.build_features_batch(
    current_season_df=all_qualified_hitters,
    historical_df=hitter_processed,
    injury_df=injury_data_historical
)

# Run complete ROS prediction workflow
from new_pipeline.common.projections.ros_projections import run_ros_predictions

results_hitters = run_ros_predictions(
    ensemble=hitter_ros,
    current_df=hitter_ros_features_all,
    historical_df=hitter_with_features,
    player_type='hitter',
    role='hitter',
    season_pct=season_pct,
    blend_ratio=0.70,
    team_games_dict=team_games_dict,
    league_median_games=league_median_games,
    use_percentile_tiers=True
)

# Unpack results
ros_predictions_all = results_hitters['predictions']
tier_labels_hitters_all = results_hitters['tiers']
projected_remaining_pa_all = results_hitters['usage_projected']
ros_display_all = results_hitters['display']

# Get thresholds for display
# Get percentile targets and actual cutoffs
from new_pipeline.models.ros.tier_thresholds import get_tier_percentiles
elite_pct, good_pct = get_tier_percentiles('hitter')
elite_cutoff = np.percentile(ros_predictions_all['mean'], 100 * (1 - elite_pct))
good_cutoff = np.percentile(ros_predictions_all['mean'], 100 * (1 - good_pct))

print(f"Tier Classification (Percentile-Based):")
print(f"  Elite: Top {elite_pct*100:.1f}% (~{int(len(all_qualified_hitters) * elite_pct)} of {len(all_qualified_hitters)} players)")
print(f"    -> Cutoff this run: {elite_cutoff:.2f} ROS WAR")
print(f"  Good: Top {good_pct*100:.1f}% (~{int(len(all_qualified_hitters) * good_pct)} of {len(all_qualified_hitters)} players)")
print(f"    -> Cutoff this run: {good_cutoff:.2f} ROS WAR")
print()


# Extract additional usage stats for display
hitter_team_games_all = all_qualified_hitters['Team'].map(team_games_dict).fillna(league_median_games).values
pa_per_game_all = all_qualified_hitters['PA'].values / np.maximum(all_qualified_hitters['G'].values, 1)
participation_rate_all = all_qualified_hitters['PA'].values / (all_qualified_hitters['G'].values * 4.5)  # Assume ~4.5 PA per game
current_pa_all = all_qualified_hitters['PA'].values


# Sort by ROS_WAR (descending) and take top 20
hitter_ros_ranking = np.argsort(ros_display_all['ros_war'])[::-1]
top_20_hitter_idx = hitter_ros_ranking[:20]

# Apply sorting to create top 20 arrays
top_20_hitters = all_qualified_hitters.iloc[top_20_hitter_idx].reset_index(drop=True)
tier_labels_hitters = tier_labels_hitters_all[top_20_hitter_idx]
projected_remaining_pa = projected_remaining_pa_all[top_20_hitter_idx]
hitter_team_games = hitter_team_games_all[top_20_hitter_idx]
pa_per_game = pa_per_game_all[top_20_hitter_idx]
participation_rate = participation_rate_all[top_20_hitter_idx]
current_pa = current_pa_all[top_20_hitter_idx]
ros_display = {
    'ros_war': ros_display_all['ros_war'][top_20_hitter_idx],
    'ros_rate': ros_display_all['ros_rate'][top_20_hitter_idx],
    'ros_q50': ros_display_all['ros_q50'][top_20_hitter_idx],
    'ros_q90': ros_display_all['ros_q90'][top_20_hitter_idx]
}

print(f"  Predicted all {len(all_qualified_hitters)} hitters, selected top 20 by ROS_WAR")
print()
print("Feature building & prediction complete!")
print("="*90)

HITTER ROS FEATURE BUILDING & PREDICTION



NameError: name 'load_current_season_data' is not defined

In [ ]:
# Cell 9.6.1: Hitter ROS Diagnostic Table

# Reload table_utils to get latest version
import importlib
import new_pipeline.notebooks.shared.table_utils
importlib.reload(new_pipeline.notebooks.shared.table_utils)

from new_pipeline.notebooks.shared.table_utils import create_ros_diagnostic_table

print("="*90)
print("HITTERS (Top 20 by Projected ROS WAR)")
print("="*90)
print()

# Create diagnostic table
hitter_table = create_ros_diagnostic_table(
    names=top_20_hitters['Name'].values.tolist(),
    teams=top_20_hitters['Team'].values.tolist(),
    tiers=tier_labels_hitters.tolist(),
    usage_current=current_pa.tolist(),
    usage_projected=projected_remaining_pa.tolist(),
    ros_display={
        'ros_war': ros_display['ros_war'].tolist(),
        'ros_rate': ros_display['ros_rate'].tolist(),
        'ros_q50': ros_display['ros_q50'].tolist(),
        'ros_q90': ros_display['ros_q90'].tolist()
    },
    player_type='hitter',
    additional_cols={
        'G': top_20_hitters['G'].values.tolist(),
        'TeamG': hitter_team_games.tolist()
    }
)

print(hitter_table)
print()

# Tier distribution summary
elite_count = (tier_labels_hitters == 'elite').sum()
good_count = (tier_labels_hitters == 'good').sum()
avg_count = (tier_labels_hitters == 'average').sum()

print(f"Tier Distribution: Elite={elite_count}, Good={good_count}, Average={avg_count}")
print()
print("="*90)